In [ ]:
import kagglehub

# Download latest version
path_r = kagglehub.dataset_download("guy007/nutrientdeficiencysymptomsinrice")

print("Path to dataset files:", path_r)

100%|██████████| 989M/989M [00:07<00:00, 135MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/guy007/nutrientdeficiencysymptomsinrice/versions/1


In [ ]:
path_m = kagglehub.dataset_download("ashishpatelresearch/maize-plant-leaf-nutrient-deficiency-dataset")

print("Path to dataset files:", path_m)

100%|██████████| 343M/343M [00:02<00:00, 167MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ashishpatelresearch/maize-plant-leaf-nutrient-deficiency-dataset/versions/1


In [ ]:
import os
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

# --- CONFIGURATION ---
# Change this to your actual folder path
base_path = 'drive/MyDrive/nutrient-deficiency/rice_1'
target_total = 1000

# Subfolders to process
folders = ['Nitrogen(N)', 'Phosphorus(P)', 'Potassium(K)']

# --- AUGMENTATION SETTINGS ---
# Keeping settings moderate to preserve nutrient deficiency features
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='reflect'
)

for folder_name in folders:
    folder_path = os.path.join(base_path, folder_name)

    if not os.path.exists(folder_path):
        print(f"Skipping {folder_name}: Folder not found.")
        continue

    # Get only the original images (ignore existing augmented ones if you run this twice)
    current_images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    num_current = len(current_images)

    if num_current >= target_total:
        print(f"✅ {folder_name} already has {num_current} images.")
        continue

    num_needed = target_total - num_current
    print(f"🔄 {folder_name}: Found {num_current}, generating {num_needed} more...")

    generated_count = 0
    while generated_count < num_needed:
        # Pick a random original image to augment
        img_name = random.choice(current_images)
        img_path = os.path.join(folder_path, img_name)

        try:
            img = load_img(img_path)
            x = img_to_array(img)
            x = x.reshape((1,) + x.shape)

            # Flow generates 1 image and saves it back to the SAME folder
            for batch in datagen.flow(x, batch_size=1,
                                      save_to_dir=folder_path,
                                      save_prefix='aug',
                                      save_format='jpg'):
                generated_count += 1
                break
        except Exception as e:
            print(f"Could not process {img_name}: {e}")
            continue

print("\n✨ All folders updated to 1000 images.")

🔄 Nitrogen(N): Found 440, generating 560 more...
🔄 Phosphorus(P): Found 333, generating 667 more...
🔄 Potassium(K): Found 383, generating 617 more...

✨ All folders updated to 1000 images.


In [ ]:
import os
import pandas as pd

def find_csv_file(directory):
    """Recursively finds the first CSV file in a directory."""
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith('.csv'):
                return os.path.join(root, file)
    return None

# Try to find and load the CSV for path_r
csv_path_r = find_csv_file(path_r)
if csv_path_r:
    print(f"Loading data from: {csv_path_r}")
    df1 = pd.read_csv(csv_path_r)
    print("Head of df1:")
    display(df1.head())
else:
    print(f"No CSV file found in {path_r}. Please specify which file to load.")
    df1 = None

# Try to find and load the CSV for path_m
csv_path_m = find_csv_file(path_m)
if csv_path_m:
    print(f"\nLoading data from: {csv_path_m}")
    df2 = pd.read_csv(csv_path_m)
    print("Head of df2:")
    display(df2.head())
else:
    print(f"\nNo CSV file found in {path_m}. Please specify which file to load.")
    df2 = None


No CSV file found in /root/.cache/kagglehub/datasets/guy007/nutrientdeficiencysymptomsinrice/versions/1. Please specify which file to load.

No CSV file found in /root/.cache/kagglehub/datasets/ashishpatelresearch/maize-plant-leaf-nutrient-deficiency-dataset/versions/1. Please specify which file to load.


In [ ]:
# center crop for coffee images belonging to flowering stage

import cv2
import os
from pathlib import Path

def smart_crop_to_leaf(input_folder, output_folder, target_size=224):
    """
    Takes 'Tree' images and crops the center to get 'Leaf-level' detail.
    """
    path = Path(input_folder)
    out_path = Path(output_folder)
    out_path.mkdir(parents=True, exist_ok=True)

    for img_file in path.glob("*.jpg"): # or *.png
        img = cv2.imread(str(img_file))
        if img is None: continue

        h, w, _ = img.shape
        # Calculate center coordinates
        center_x, center_y = w // 2, h // 2

        # We want to take a square that is roughly 40-50% of the tree size
        # to ensure we get a cluster of leaves/berries
        crop_width = int(min(h, w) * 0.5)

        start_x = max(0, center_x - crop_width // 2)
        start_y = max(0, center_y - crop_width // 2)

        # Slicing the image (Crop)
        cropped = img[start_y:start_y+crop_width, start_x:start_x+crop_width]

        # Resize to your final model input size
        final_img = cv2.resize(cropped, (target_size, target_size))

        cv2.imwrite(str(out_path / img_file.name), final_img)

# Example usage for your Stage 3 Coffee Tree images
smart_crop_to_leaf('Coffee/3_Flowering_Trees', 'Coffee/3_Flowering_Leaves')
print("Cropping complete! Your coffee images now match the scale of Rice/Maize.")

Cropping complete! Your coffee images now match the scale of Rice/Maize.


In [ ]:
# data augmentation for coffee seedling stage images
import albumentations as A
import cv2
import os
import random
from pathlib import Path

# --- CONFIGURATION ---
INPUT_DIR = "Coffee/Stage1_Raw"  # Folder with your few seedling images
OUTPUT_DIR = "Coffee/1"          # Folder where 1000 images will go
TOTAL_IMAGES_NEEDED = 1000

# Define the "Variations" (Tilts, Flips, Brightness)
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.RGBShift(r_shift_limit=10, g_shift_limit=10, b_shift_limit=10, p=0.3),
])

os.makedirs(OUTPUT_DIR, exist_ok=True)
images = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

print(f"Generating images... target: {TOTAL_IMAGES_NEEDED}")

for i in range(TOTAL_IMAGES_NEEDED):
    # Pick a random image from your small set
    image_name = random.choice(images)
    image = cv2.imread(os.path.join(INPUT_DIR, image_name))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Apply the magic
    augmented = transform(image=image)["image"]

    # Save the new version
    save_path = os.path.join(OUTPUT_DIR, f"seedling_aug_{i}.jpg")
    cv2.imwrite(save_path, cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))

print(f"Done! Check {OUTPUT_DIR} for your 1,000 images.")

In [ ]:
import os
import cv2
import random
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

# --- CONFIGURATION ---
# Set this to your specific Coffee P folder path
folder_path = 'drive/MyDrive/nutrient-deficiency/coffee_deficiency/phosphorus-P'
target_total = 1000
save_prefix = 'aug_p'

# 1. Define Augmentation Strategy
# We use 'reflect' to keep the leaf edges natural for Phosphorus identification
datagen = ImageDataGenerator(
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='reflect'
)

# 2. Identify current images
image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
num_current = len(image_files)

if num_current >= target_total:
    print(f"✅ Folder already has {num_current} images. No action taken.")
else:
    num_needed = target_total - num_current
    print(f"🚀 Current: {num_current} | Goal: {target_total} | Generating: {num_needed}")

    generated_count = 0

    # Loop through original images to create variations
    while generated_count < num_needed:
        for filename in image_files:
            if generated_count >= num_needed:
                break

            img_path = os.path.join(folder_path, filename)

            try:
                # Load image
                img = load_img(img_path)
                x = img_to_array(img)
                x = x.reshape((1,) + x.shape) # Reshape for the generator

                # Generate and Save
                # flow() saves directly to the folder_path
                for batch in datagen.flow(x, batch_size=1,
                                          save_to_dir=folder_path,
                                          save_prefix=save_prefix,
                                          save_format='jpg'):
                    generated_count += 1
                    break # Move to the next seed image
            except Exception as e:
                print(f"Error processing {filename}: {e}")
                continue

    print(f"✨ Success! {folder_path} now contains {len(os.listdir(folder_path))} total images.")

🚀 Current: 354 | Goal: 1000 | Generating: 646
✨ Success! drive/MyDrive/nutrient-deficiency/coffee_deficiency/phosphorus-P now contains 983 total images.


In [ ]:
import os

# --- CONFIGURATION ---
# Replace with your folder path (e.g., 'coffee_p_deficiency')
folder_path = 'drive/MyDrive/nutrient-deficiency/coffee_deficiency/phosphorus-P'

def count_images(path):
    # List of common image extensions
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

    if not os.path.exists(path):
        return "Folder not found."

    # Count files that end with the valid extensions
    files = [f for f in os.listdir(path) if f.lower().endswith(valid_extensions)]
    return len(files)

# Execute
image_count = count_images(folder_path)
print(f"📊 Total images in folder: {image_count}")

📊 Total images in folder: 977


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')

# --- SETTINGS ---
BASE_PATH = '/content/drive/MyDrive/nutrient-deficiency'
IMG_SIZE = 224
BATCH_SIZE = 32

crop_map = {'rice_deficiency': 0, 'maize_deficiency': 1, 'coffee_deficiency': 2}
label_map = {'Nitrogen(N)': 0, 'Phosphorus(P)': 1, 'Potassium(K)': 2, 'Healthy': 3}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Create the Master List
data = []
for crop_folder, crop_id in crop_map.items():
    crop_path = os.path.join(BASE_PATH, crop_folder)
    for label_name, label_id in label_map.items():
        f_path = os.path.join(crop_path, label_name)
        if os.path.exists(f_path):
            for img in os.listdir(f_path):
                if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                    data.append([os.path.join(f_path, img), crop_id, label_id])

df = pd.DataFrame(data, columns=['filepath', 'crop_id', 'label_id'])
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label_id'], random_state=42)

# Training Dataset Pipeline
train_ds = tf.data.Dataset.from_tensor_slices((train_df['filepath'], train_df['crop_id'], train_df['label_id']))
train_ds = train_ds.map(lambda f, c, l: (
    (tf.image.resize(tf.image.decode_image(tf.io.read_file(f), channels=3, expand_animations=False), [IMG_SIZE, IMG_SIZE]) / 255.0, tf.one_hot(c, 3)),
    tf.one_hot(l, 4)
), num_parallel_calls=tf.data.AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Validation Dataset Pipeline
val_ds = tf.data.Dataset.from_tensor_slices((val_df['filepath'], val_df['crop_id'], val_df['label_id']))
val_ds = val_ds.map(lambda f, c, l: (
    (tf.image.resize(tf.image.decode_image(tf.io.read_file(f), channels=3, expand_animations=False), [IMG_SIZE, IMG_SIZE]) / 255.0, tf.one_hot(c, 3)),
    tf.one_hot(l, 4)
)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
import numpy as np
from PIL import Image

# --- 1. THE SAFE GENERATOR (Captures corrupt files & keeps training alive) ---
def safe_generator(df_partition):
    for _, row in df_partition.iterrows():
        try:
            # Attempt to open and process using PIL for safety
            img = Image.open(row['filepath']).convert('RGB')
            img = img.resize((IMG_SIZE, IMG_SIZE))
            img_array = np.array(img) / 255.0

            # Metadata Input (One-hot for Rice, Maize, Coffee)
            crop_oh = np.zeros(3)
            crop_oh[row['crop_id']] = 1

            # Target Output (One-hot for N, P, K, Healthy)
            label_oh = np.zeros(4)
            label_oh[row['label_id']] = 1

            yield (img_array, crop_oh), label_oh

        except Exception:
            # This handles the "Uncompress failed" error by skipping the file
            print(f" SKIPPING CORRUPT FILE: {row['filepath']}")
            continue

# --- 2. CONVERT TO TF DATASETS ---
sig = ((tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(3,), dtype=tf.float32)),
       tf.TensorSpec(shape=(4,), dtype=tf.float32))

train_ds = tf.data.Dataset.from_generator(lambda: safe_generator(train_df), output_signature=sig).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_generator(lambda: safe_generator(val_df), output_signature=sig).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# --- 3. ARCHITECTURE ---
img_in = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")
crop_in = layers.Input(shape=(3,), name="crop_input")

# EfficientNetV2S Backbone
base = tf.keras.applications.EfficientNetV2S(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False

x = layers.GlobalAveragePooling2D()(base(img_in))
x = layers.Dropout(0.3)(x)

# Fusion Logic
y = layers.Dense(16, activation='relu')(crop_in)
merged = layers.Concatenate()([x, y])
z = layers.Dense(256, activation='relu')(merged)
z = layers.Dropout(0.4)(z)
out = layers.Dense(4, activation='softmax')(z)

model = models.Model(inputs=[img_in, crop_in], outputs=out)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# --- 4. EXECUTION ---
print(" Stage 1: Warm-up (10 Epochs)")
model.fit(train_ds, validation_data=val_ds, epochs=10)

#print(" Stage 2: Fine-Tuning (20 Epochs)")
# Note: In the Functional API, the base model is usually model.layers[2]
#model.layers[2].trainable = True
#model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
#model.fit(train_ds, validation_data=val_ds, epochs=20)



 Stage 1: Warm-up (10 Epochs)
Epoch 1/10
     68/Unknown 554s 7s/step - accuracy: 0.3298 - loss: 1.3160 SKIPPING CORRUPT FILE: /content/drive/MyDrive/nutrient-deficiency/coffee_deficiency/Healthy/2 (691).jpg
    299/Unknown 3833s 13s/step - accuracy: 0.4181 - loss: 1.2022

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


299/299 ━━━━━━━━━━━━━━━━━━━━ 4978s 16s/step - accuracy: 0.4817 - loss: 1.1082 - val_accuracy: 0.6127 - val_loss: 0.9038
Epoch 2/10
 68/299 ━━━━━━━━━━━━━━━━━━━━ 25:42 7s/step - accuracy: 0.5415 - loss: 0.9836 SKIPPING CORRUPT FILE: /content/drive/MyDrive/nutrient-deficiency/coffee_deficiency/Healthy/2 (691).jpg
299/299 ━━━━━━━━━━━━━━━━━━━━ 2518s 8s/step - accuracy: 0.5641 - loss: 0.9489 - val_accuracy: 0.6467 - val_loss: 0.8116
Epoch 3/10
 68/299 ━━━━━━━━━━━━━━━━━━━━ 25:08 7s/step - accuracy: 0.5782 - loss: 0.9107 SKIPPING CORRUPT FILE: /content/drive/MyDrive/nutrient-deficiency/coffee_deficiency/Healthy/2 (691).jpg
299/299 ━━━━━━━━━━━━━━━━━━━━ 2511s 8s/step - accuracy: 0.5921 - loss: 0.8909 - val_accuracy: 0.6521 - val_loss: 0.7874
Epoch 4/10
 68/299 ━━━━━━━━━━━━━━━━━━━━ 25:25 7s/step - accuracy: 0.5919 - loss: 0.8984 SKIPPING CORRUPT FILE: /content/drive/MyDrive/nutrient-deficiency/coffee_deficiency/Healthy/2 (691).jpg
299/299 ━━━━━━━━━━━━━━━━━━━━ 2455s 8s/step - accuracy: 0.5994 - lo

KeyboardInterrupt: 

In [ ]:
model.save('deficiency_power_model.keras')
print(" Done! Your power model is saved.")

 Done! Your power model is saved.


In [ ]:
import tensorflow as tf
from tensorflow.keras import models

# Load the saved model from Stage 1
model = models.load_model('deficiency_power_model.keras')
print("✅ Model loaded successfully from Stage 1")

✅ Model loaded successfully from Stage 1


In [ ]:
data = []
for crop_folder, crop_id in crop_map.items():
    crop_path = os.path.join(BASE_PATH, crop_folder)
    for label_name, label_id in label_map.items():
        f_path = os.path.join(crop_path, label_name)
        if os.path.exists(f_path):
            for img in os.listdir(f_path):
                if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                    data.append([os.path.join(f_path, img), crop_id, label_id])

df = pd.DataFrame(data, columns=['filepath', 'crop_id', 'label_id'])
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label_id'], random_state=42)

In [ ]:
import numpy as np
from PIL import Image

# Re-run the generator so the model has data
def safe_generator(df_partition):
    for _, row in df_partition.iterrows():
        try:
            img = Image.open(row['filepath']).convert('RGB')
            img = img.resize((IMG_SIZE, IMG_SIZE))
            img_array = np.array(img) / 255.0
            crop_oh = np.zeros(3); crop_oh[row['crop_id']] = 1
            label_oh = np.zeros(4); label_oh[row['label_id']] = 1
            yield (img_array, crop_oh), label_oh
        except Exception:
            print(f" ⚠️ SKIPPING CORRUPT FILE: {row['filepath']}")
            continue

sig = ((tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(3,), dtype=tf.float32)),
       tf.TensorSpec(shape=(4,), dtype=tf.float32))

train_ds = tf.data.Dataset.from_generator(lambda: safe_generator(train_df), output_signature=sig).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_generator(lambda: safe_generator(val_df), output_signature=sig).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# 1. Load the model you saved after Stage 1
model = tf.keras.models.load_model('deficiency_power_model.keras')

print(" Resuming for Stage 2: Fine-Tuning")

# 2. Unfreeze the backbone (EfficientNet)
model.layers[2].trainable = True

# 3. Re-compile with the LOW learning rate (Crucial!)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# 4. Start the 20 epochs
model.fit(train_ds, validation_data=val_ds, epochs=20)

model.save('deficiency_power_model_final.keras')
print(" Done! Final fine-tuned model saved.")

 Resuming for Stage 2: Fine-Tuning
Epoch 1/20
     69/Unknown 1257s 12s/step - accuracy: 0.6374 - loss: 0.8018 ⚠️ SKIPPING CORRUPT FILE: /content/drive/MyDrive/nutrient-deficiency/coffee_deficiency/Healthy/2 (691).jpg
    299/Unknown 3819s 11s/step - accuracy: 0.6419 - loss: 0.7810

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


299/299 ━━━━━━━━━━━━━━━━━━━━ 4726s 14s/step - accuracy: 0.6480 - loss: 0.7675 - val_accuracy: 0.7045 - val_loss: 0.6788
Epoch 2/20
 68/299 ━━━━━━━━━━━━━━━━━━━━ 24:28 6s/step - accuracy: 0.6567 - loss: 0.7741 ⚠️ SKIPPING CORRUPT FILE: /content/drive/MyDrive/nutrient-deficiency/coffee_deficiency/Healthy/2 (691).jpg
299/299 ━━━━━━━━━━━━━━━━━━━━ 2351s 8s/step - accuracy: 0.6576 - loss: 0.7564 - val_accuracy: 0.7020 - val_loss: 0.6758
Epoch 3/20
 68/299 ━━━━━━━━━━━━━━━━━━━━ 23:52 6s/step - accuracy: 0.6362 - loss: 0.7787 ⚠️ SKIPPING CORRUPT FILE: /content/drive/MyDrive/nutrient-deficiency/coffee_deficiency/Healthy/2 (691).jpg
299/299 ━━━━━━━━━━━━━━━━━━━━ 2323s 8s/step - accuracy: 0.6520 - loss: 0.7514 - val_accuracy: 0.7037 - val_loss: 0.6739
Epoch 4/20
 68/299 ━━━━━━━━━━━━━━━━━━━━ 24:09 6s/step - accuracy: 0.6686 - loss: 0.7680 ⚠️ SKIPPING CORRUPT FILE: /content/drive/MyDrive/nutrient-deficiency/coffee_deficiency/Healthy/2 (691).jpg
299/299 ━━━━━━━━━━━━━━━━━━━━ 2357s 8s/step - accuracy: 0.

In [ ]:
for i, layer in enumerate(model.layers):
    print(f"Index {i}: {layer.name} - {type(layer)}")

Index 0: image_input - <class 'keras.src.layers.core.input_layer.InputLayer'>
Index 1: efficientnetv2-s - <class 'keras.src.models.functional.Functional'>
Index 2: global_average_pooling2d_1 - <class 'keras.src.layers.pooling.global_average_pooling2d.GlobalAveragePooling2D'>
Index 3: crop_input - <class 'keras.src.layers.core.input_layer.InputLayer'>
Index 4: dropout_2 - <class 'keras.src.layers.regularization.dropout.Dropout'>
Index 5: dense_3 - <class 'keras.src.layers.core.dense.Dense'>
Index 6: concatenate_1 - <class 'keras.src.layers.merging.concatenate.Concatenate'>
Index 7: dense_4 - <class 'keras.src.layers.core.dense.Dense'>
Index 8: dropout_3 - <class 'keras.src.layers.regularization.dropout.Dropout'>
Index 9: dense_5 - <class 'keras.src.layers.core.dense.Dense'>


In [ ]:
# 1. Access the EfficientNet backbone at Index 1
base_model = model.layers[1]

# 2. Unfreeze the backbone
base_model.trainable = True

# 3. Freeze all layers EXCEPT the last 20 layers of the backbone
# This keeps the low-level feature detectors (edges/shapes) stable
for layer in base_model.layers[:-20]:
    layer.trainable = False

# 4. Re-compile with the fine-tuning learning rate
# Using a low rate (1e-5) is vital so we don't wreck the pre-trained weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Stage 2: Backbone partially unfrozen. Ready to train.")

Stage 2: Backbone partially unfrozen. Ready to train.


In [ ]:
import tensorflow as tf

# Define where to save the best version on your Drive
checkpoint_path = "/content/drive/MyDrive/nutrient-deficiency/best_fine_tuned_model.keras"

cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

# Start training
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[cp_callback]
)

# Final manual save
model.save('deficiency_power_model_v2_final.keras')
print("Done! Final fine-tuned model saved.")

Epoch 1/20
     67/Unknown 1111s 11s/step - accuracy: 0.5642 - loss: 1.4044

In [ ]:
from datasets import load_dataset

ds = load_dataset("mohanty/PlantVillage")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ValueError: BuilderConfig 'color' not found. Available: ['default']

In [ ]:
import shutil
import os
from google.colab import drive

# 1. Mount your Drive if you haven't already
drive.mount('/content/drive')

# 2. Define your source (Colab) and destination (Drive)
# Change 'SEARCH_TERM' to the folder name the downloader created
source_folder = '/content/temp_images/maize seedling V1 stage'
destination_folder = '/content/drive/MyDrive/growth-stages/maize/1'

# 3. Create the destination folder if it doesn't exist
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)

# 4. Move all images
files = os.listdir(source_folder)
for f in files:
    shutil.move(os.path.join(source_folder, f), os.path.join(destination_folder, f))

print(f"✅ Moved {len(files)} images to your Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/temp_images/maize seedling V1 stage'

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# --- 1. IMAGE INPUT BRANCH ---
img_input = layers.Input(shape=(224, 224, 3), name="image_input")

# Load EfficientNetV2-S without the top classification layers
base_model = tf.keras.applications.EfficientNetV2S(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Keep the pre-trained weights frozen for now

# Process image features
x = base_model(img_input)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)

# --- 2. CROP ID INPUT BRANCH ---
# Input for Crop Type: Rice, Maize, Coffee (One-hot encoded, length 3)
crop_input = layers.Input(shape=(3,), name="crop_input")
y = layers.Dense(16, activation='relu')(crop_input)

# --- 3. THE FUSION ---
# Merging the visual features with the crop metadata
combined = layers.Concatenate()([x, y])

# Final Classification Head
z = layers.Dense(128, activation='relu')(combined)
z = layers.Dropout(0.3)(z)

# Output: 3 Classes (Stage 1, 2, or 3)
output = layers.Dense(3, activation='softmax', name="stage_output")(z)

# --- 4. ASSEMBLE & COMPILE ---
growth_model = models.Model(inputs=[img_input, crop_input], outputs=output)

growth_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Print the architecture to verify the connections
growth_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetv2-s    │ (None, 7, 7,      │ 20,331,360 │ image_input[0][0] │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ efficientnetv2-s… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    327,936 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ crop_input          │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │         64 │ crop_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 272)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     34,944 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage_output        │ (None, 3)         │        387 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,694,691 (78.94 MB)

 Trainable params: 363,331 (1.39 MB)

 Non-trainable params: 20,331,360 (77.56 MB)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# 1. Define where to save the best version of your Growth Model
checkpoint_path = '/content/drive/MyDrive/nutrient-deficiency/best_growth_fusion_model.keras'

# 2. Set up Callbacks for "Smart" Training
callbacks = [
    # Saves only the best model based on validation accuracy
    ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_accuracy', mode='max'),

    # Stops training if the model stops improving (prevents overfitting)
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),

    # Lowers learning rate if progress plateaus (helps fine-tune the weights)
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
]

# 3. Start the Training
# Note: Ensure train_gen and val_gen from the previous step are initialized
history = growth_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25, # Starting with 25; EarlyStopping will kill it if it finishes early
    callbacks=callbacks,
    verbose=1
)

print(f" Training Complete! Best model saved to: {checkpoint_path}")

Epoch 1/25
224/224 ━━━━━━━━━━━━━━━━━━━━ 1847s 8s/step - accuracy: 0.5150 - loss: 0.9592 - val_accuracy: 0.5952 - val_loss: 0.8029 - learning_rate: 0.0010
Epoch 2/25
224/224 ━━━━━━━━━━━━━━━━━━━━ 144s 645ms/step - accuracy: 0.6004 - loss: 0.8404 - val_accuracy: 0.6873 - val_loss: 0.7185 - learning_rate: 0.0010
Epoch 3/25
224/224 ━━━━━━━━━━━━━━━━━━━━ 139s 617ms/step - accuracy: 0.6267 - loss: 0.7909 - val_accuracy: 0.6801 - val_loss: 0.6895 - learning_rate: 0.0010
Epoch 4/25
224/224 ━━━━━━━━━━━━━━━━━━━━ 125s 555ms/step - accuracy: 0.6407 - loss: 0.7659 - val_accuracy: 0.7046 - val_loss: 0.6509 - learning_rate: 0.0010
Epoch 5/25
224/224 ━━━━━━━━━━━━━━━━━━━━ 123s 549ms/step - accuracy: 0.6553 - loss: 0.7446 - val_accuracy: 0.7108 - val_loss: 0.6441 - learning_rate: 0.0010
Epoch 6/25
224/224 ━━━━━━━━━━━━━━━━━━━━ 120s 536ms/step - accuracy: 0.6726 - loss: 0.7168 - val_accuracy: 0.7549 - val_loss: 0.5939 - learning_rate: 0.0010
Epoch 7/25
224/224 ━━━━━━━━━━━━━━━━━━━━ 157s 606ms/step - accuracy

In [ ]:
model.save('growth_model.keras')

#run from here

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

all_data = []
BASE_PATH = '/content/drive/MyDrive/nutrient-deficiency'
CROP_FOLDERS = ['rice_deficiency', 'maize_deficiency', 'coffee_deficiency']
DEFICIENCY_LABELS = ['Potassium(K)', 'Phosphorus(P)', 'Nitrogen(N)', 'Healthy']

for crop_id, crop_name in enumerate(CROP_FOLDERS):
    for label_id, def_name in enumerate(DEFICIENCY_LABELS):
        folder_path = os.path.join(BASE_PATH, crop_name, def_name)
        if os.path.exists(folder_path):
            images = [os.path.join(folder_path, f) for f in os.listdir(folder_path)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            for img_path in images:
                all_data.append({'filepath': img_path, 'crop_id': crop_id, 'label_id': label_id})

df = pd.DataFrame(all_data)
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df[['crop_id', 'label_id']], random_state=42)
print(f"✅ Dataset Loaded: {len(df)} images with 4 classes.")

✅ Dataset Loaded: 11927 images with 4 classes.


In [3]:
import tensorflow as tf

IMG_SIZE = 300

def process_dual_input(filepath, crop_id, label_id, augment=True):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.1)
    img = img / 255.0

    # Dictionary keys must match the name="" arguments in your Input layers
    return {"image_input": img, "crop_input": tf.one_hot(crop_id, 3)}, tf.one_hot(label_id, 4)

train_gen = tf.data.Dataset.from_tensor_slices((train_df['filepath'], train_df['crop_id'], train_df['label_id']))\
    .shuffle(len(train_df)).map(lambda f, c, l: process_dual_input(f, c, l, True)).batch(16).prefetch(tf.data.AUTOTUNE)

val_gen = tf.data.Dataset.from_tensor_slices((val_df['filepath'], val_df['crop_id'], val_df['label_id']))\
    .map(lambda f, c, l: process_dual_input(f, c, l, False)).batch(16).prefetch(tf.data.AUTOTUNE)

In [4]:
from tensorflow.keras import layers, models

# 1. Create the 300px Shell
img_in = layers.Input(shape=(300, 300, 3), name="image_input")
crop_in = layers.Input(shape=(3,), name="crop_input")

base = tf.keras.applications.EfficientNetV2S(input_tensor=img_in, include_top=False, weights='imagenet')
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dense(256, activation='relu')(x)
crop_branch = layers.Dense(16, activation='relu')(crop_in)
combined = layers.Concatenate()([x, crop_branch])
output = layers.Dense(4, activation='softmax')(combined)

def_model_300 = models.Model(inputs=[img_in, crop_in], outputs=output)

# 2. Transfer weights from your 72% .keras file
old_model = tf.keras.models.load_model('/content/drive/MyDrive/nutrient-deficiency/best_fine_tuned_model.keras')
for i in range(len(old_model.layers)):
    try:
        def_model_300.layers[i].set_weights(old_model.layers[i].get_weights())
    except:
        continue # Skips input layer mismatch

def_model_300.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
print("✅ Knowledge transferred to 300px shell.")

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
✅ Knowledge transferred to 300px shell.


In [5]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

check_path = '/content/drive/MyDrive/nutrient-deficiency/deficiency_300px_autosave.keras'

callbacks = [
    ModelCheckpoint(check_path, save_best_only=True, monitor='val_accuracy', mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7, verbose=1)
]

print("🚀 Starting training. Auto-save is active.")
history = def_model_300.fit(train_gen, validation_data=val_gen, epochs=25, callbacks=callbacks)

🚀 Starting training. Auto-save is active.
Epoch 1/25


KeyboardInterrupt: 